# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brentjoel/FlyRank-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 4 — CTR / Engagement Opportunity. Task type: scoring, delivered as a ranking.**

Every visible page gets an **opportunity score** = *how many clicks it is missing compared with what a similar page normally gets*. Pages are then sorted by that score, and the editor works from the top.

- **Why scoring/ranking, not plain classification:** an editor has time for maybe 20–50 pages a week, so the useful output is an *order* ("fix these first"), not a yes/no for 13,000 pages. A classifier can still sit inside the score (probability that the gap is real and persists), but what ships is a ranked list.
- **Why not clustering:** I already know what I am looking for (under-clicked pages). Clustering could help later to group *why* pages under-perform, but it is not the main task.

**My lane on the ML loop:**

| Loop step | In my lane |
|---|---|
| Data | starter CSV now (one row per page, trailing 90 days); warehouse daily table from Week 3 |
| Unit | one visible page (content item) in one time window |
| Inputs | impressions, clicks, position, intent, content type, age/freshness, volume — from the *earlier* window only |
| Target / proxy | observed CTR gap vs. comparable pages, checked in the *later* window (section 2) |
| Output | opportunity score + rank + reason codes |
| Metric | precision@50 on "gap persists", vs. a fixed-rule baseline (section 3) |
| **Action** | editor reviews the top of the queue: rewrite title/meta, fix intent match, improve snippet structure, or mark *monitor* |
| Feedback | editor's verdict on each page + CTR after the change → better labels next round |

**What the output supports (output → content action):**

| Reason code on the queue | Suggested action |
|---|---|
| strong position + high impressions + CTR far below peers | rewrite title and meta description |
| CTR low for the page's intent (e.g. transactional page acting like an info page) | improve intent match / page angle |
| CTR low but clicks recovering in the last 30 days | monitor, don't touch |
| many clicks but weak engagement after the click | improve on-page content (secondary, engagement side of the lane) |

> **My frame in one paragraph:** For SEO/content editors deciding *which visible pages to review first*, I will build a **ranked opportunity score** from page-level search data, scoring **how far each page's observed CTR sits below comparable pages** (same position range, intent, content type). It is measured by **precision@50**: of the top 50 pages, how many still show the gap in the next 30 days (and, later, how many an editor agrees are worth fixing). A wrong call costs a wasted editor hour (false positive) or keeps losing clicks on a page that already ranks (false negative). A plain rule isn't enough because expected CTR depends on several tangled signals at once (section 5). I will claim only observed, directional, decision-support results.

In [1]:
import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brentjoel/FlyRank-1"
REPO_DIR = "FlyRank-1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    here = Path.cwd()
    while not (here / "data" / "raw").is_dir() and here.parent != here:
        here = here.parent
    os.chdir(here)

DATA = Path("data/raw/content_refresh_anonymized.csv")
assert DATA.exists(), "starter CSV not found - are you inside the repo?"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

df = pd.read_csv(DATA)
print(f"starter data: {len(df):,} pages x {df.shape[1]} columns, {df['client_id'].nunique()} clients")

# My lane's slice: pages that are actually VISIBLE in search, with enough volume to judge CTR.
MIN_IMPR_90D = 1_000                       # volume floor from ML-02: below this, CTR is noise
has_pos = df["avg_position"] > 0           # avg_position == 0 means "no data", not rank 0
lane = df[has_pos & (df["impressions_90d"] >= MIN_IMPR_90D)].copy()
print(f"lane slice (position data + >= {MIN_IMPR_90D:,} impressions/90d): {len(lane):,} pages "
      f"from {lane['client_id'].nunique()} clients, holding "
      f"{lane['impressions_90d'].sum() / df['impressions_90d'].sum():.1%} of all impressions")
print("task type: SCORING -> RANKED review queue (one score per page)")

starter data: 30,000 pages x 44 columns, 32 clients
lane slice (position data + >= 1,000 impressions/90d): 13,512 pages from 28 clients, holding 97.5% of all impressions
task type: SCORING -> RANKED review queue (one score per page)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no column in the data that says "this page has a bad title". So I need a **proxy**, and I want it to be an **observed outcome in a later time window**, not a rule I invent and then teach a model to copy.

**The split in time (inside the starter data):**
- **Input window = the earlier 30 days** (`clicks_prev_30d`, `impressions_prev_30d`, days 31–60 back) + page facts (intent, content type, age, word count).
- **Outcome window = the most recent 30 days** (`clicks_last_30d`, `impressions_last_30d`).

**Two target columns (sketched in code below):**
1. **`ctr_gap_ratio_next30`** (continuous) = the page's CTR in the outcome window ÷ the pooled CTR of pages in the same position tier in that window. `1.0` = normal for its position, `0.2` = gets one fifth of the clicks a comparable page gets. This is what a model would learn to predict, and the score is built from it.
2. **`gap_persists_next30`** (yes/no, used for evaluation) = `ctr_gap_ratio_next30 < 0.5`, i.e. the page is *still* below half of its peers in the later window.

**Observed or defined?** The clicks and impressions are **observed** (they really happened in the later window). The `< 0.5` cut-off **is a convention I chose**, so I will report the result at a second cut-off too (e.g. 0.33) before trusting it. The opportunity *size* used for ranking is also observed: **missed clicks = impressions × (peer CTR − page CTR)**.

**Leakage rules I am following:**
- `trend_direction` and `trend_pct` are **never** used (they are the decline-lane label source).
- Inputs come only from the earlier window. ⚠️ Known issue: `avg_position` / `position_tier` in the starter CSV is a 90-day average that **overlaps** the outcome window. Here I use it only to *group peers*; in the warehouse (Week 3+) I will compute position from the input window only.
- Pages need ≥ 300 impressions in **each** 30-day window, otherwise one click swings the CTR.

In [2]:
MIN_IMPR_30D = 300   # per-window volume floor

t = lane[(lane["impressions_prev_30d"] >= MIN_IMPR_30D) &
         (lane["impressions_last_30d"] >= MIN_IMPR_30D)].copy()

# Input window (earlier 30 days) -- what the score is allowed to see
t["ctr_prev30"] = 100 * t["clicks_prev_30d"] / t["impressions_prev_30d"]          # in %, like `ctr`
# peer CTR = POOLED CTR of the same position tier (clicks / impressions); pooled, not median,
# because the median CTR of deep pages is 0 and would make the ratio below divide by zero
def tier_pooled_ctr(frame, clicks, impr):
    g = frame.groupby("position_tier")
    return 100 * g[clicks].transform("sum") / g[impr].transform("sum")

t["peer_ctr_prev30"] = tier_pooled_ctr(t, "clicks_prev_30d", "impressions_prev_30d")
t["missed_clicks_prev30"] = (t["impressions_prev_30d"] *
                             (t["peer_ctr_prev30"] - t["ctr_prev30"]).clip(lower=0) / 100)

# Outcome window (most recent 30 days) -- the observed target, never an input
t["ctr_next30"] = 100 * t["clicks_last_30d"] / t["impressions_last_30d"]
t["peer_ctr_next30"] = tier_pooled_ctr(t, "clicks_last_30d", "impressions_last_30d")
t["ctr_gap_ratio_next30"] = t["ctr_next30"] / t["peer_ctr_next30"]
t["gap_persists_next30"] = (t["ctr_gap_ratio_next30"] < 0.5).astype(int)

print(f"pages with >= {MIN_IMPR_30D} impressions in both windows: {len(t):,}")
print("\nctr_gap_ratio_next30 (1.0 = normal for its position):")
print(t["ctr_gap_ratio_next30"].describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_string())
for cut in (0.5, 0.33):
    print(f"\nbase rate of 'gap persists' at cut-off < {cut}: "
          f"{(t['ctr_gap_ratio_next30'] < cut).mean():.1%}")
print("\nleakage guard - trend columns used:", [c for c in ("trend_pct", "trend_direction")
                                               if c in ("ctr_prev30", "missed_clicks_prev30")])

pages with >= 300 impressions in both windows: 10,862

ctr_gap_ratio_next30 (1.0 = normal for its position):
count    10862.00
mean         1.06
std          1.47
min          0.00
10%          0.00
25%          0.27
50%          0.72
75%          1.41
90%          2.42
max         65.45

base rate of 'gap persists' at cut-off < 0.5: 37.3%

base rate of 'gap persists' at cut-off < 0.33: 27.8%

leakage guard - trend columns used: []


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@50** — rank pages using only the earlier window, take the **top 50**, and count how many of them **still show the gap in the next 30 days** (`gap_persists_next30 = 1`).

**Why this one:** editor time is the scarce thing. They will never read the whole list, so only the top of the list matters. A false positive at the top wastes a real person's hour; an error at rank 9,000 costs nothing. (Later, with editor feedback, the same metric becomes "how many of the top 20 did a human agree were worth fixing".)

**What number means "good"** (computed below, not typed by hand):
- **Floor 1 — base rate:** picking 50 pages at random would hit the base rate shown below (~37%). Anything near that is useless.
- **Floor 2 — the fixed rule:** the simple rule *"rank by missed clicks vs. the tier's typical CTR"* is my baseline. Below it scores about **82% precision@50** (vs. ~37% random and 58% for a naive lowest-CTR sort), so **the bar is high** at the very top. It drops further down the list (about 76% at 200 and 70% at 500), which is where a better score has room to help. The top-50 also only covers ~14% of next month's missed clicks, so there is a lot of opportunity below it.
- **Good =** a score that beats the fixed rule on precision@50 **when tested on clients it has never seen** (grouped split by `client_id`), and also captures at least as many missed clicks in its top 50. If a model can't beat the rule, I ship the rule — that is still a useful result.

Secondary number I will always report next to it: **share of next-30-day missed clicks captured by the top 50** (does the list point at the *big* opportunities, not just many small ones?).

In [3]:
K = 50
KS = (50, 200, 500)
base_rate = t["gap_persists_next30"].mean()

def precision_at_k(frame, score_col, k=K):
    top = frame.sort_values(score_col, ascending=False).head(k)
    return top["gap_persists_next30"].mean(), top

# Outcome-window missed clicks, used only to measure how much opportunity the top-K points at
t["missed_clicks_next30"] = (t["impressions_last_30d"] *
                             (t["peer_ctr_next30"] - t["ctr_next30"]).clip(lower=0) / 100)

# Baseline A: the fixed rule -> rank by missed clicks in the earlier window
p_rule, top_rule = precision_at_k(t, "missed_clicks_prev30")
# Baseline B: naive sort by lowest raw CTR (ignores position and volume)
t["neg_ctr_prev30"] = -t["ctr_prev30"]
p_raw, top_raw = precision_at_k(t, "neg_ctr_prev30")
# Baseline C: random order (average of 200 shuffles)
rng = np.random.default_rng(0)
p_rand = np.mean([t.sample(K, random_state=int(s))["gap_persists_next30"].mean()
                  for s in rng.integers(0, 1_000_000, 200)])

total_next = t["missed_clicks_next30"].sum()
summary = pd.DataFrame({
    "precision@50": [p_rand, p_raw, p_rule],
    "share_of_next30_missed_clicks_in_top50": [
        K / len(t),
        top_raw["missed_clicks_next30"].sum() / total_next,
        top_rule["missed_clicks_next30"].sum() / total_next],
}, index=["random", "lowest raw CTR", "fixed rule: missed clicks vs tier CTR"]).round(3)

print(f"base rate (gap persists): {base_rate:.1%}  over {len(t):,} pages")
print(summary.to_string())
print("\nfixed rule further down the list (precision@K):",
      {k: round(float(precision_at_k(t, "missed_clicks_prev30", k)[0]), 3) for k in KS})
print("\n-> 'good' for a model = beat the fixed-rule row on BOTH columns, on held-out clients.")

base rate (gap persists): 37.3%  over 10,862 pages


                                       precision@50  share_of_next30_missed_clicks_in_top50
random                                         0.37                                   0.005
lowest raw CTR                                 0.58                                   0.002
fixed rule: missed clicks vs tier CTR          0.82                                   0.137

fixed rule further down the list (precision@K): {50: 0.82, 200: 0.76, 500: 0.702}

-> 'good' for a model = beat the fixed-rule row on BOTH columns, on held-out clients.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page (content item) of one pseudonymized client, in one time window** (here: the trailing 90 days, split into an earlier and a later 30-day window).

It is *not* one search query and *not* one day. That matters: `avg_position` is an average over all the queries a page shows up for, so a page can look like "position 8" while actually ranking #2 for a small query and #40 for a big one. In the warehouse the grain becomes *page × day*, and I will roll it up back to this page-level unit.

The first table shows the unit with its key columns. The second table is the **target sketch**: inputs from the earlier window on the left, the observed outcome from the later window on the right. The third shows what the editor would actually see — the top of the fixed-rule queue with reason codes and an action.

In [4]:
# Grain check: one row per content_id, no duplicates
assert lane["content_id"].is_unique, "grain broken: duplicate content_id"
print(f"grain check passed: {len(lane):,} rows = {lane['content_id'].nunique():,} unique pages\n")

unit_cols = ["content_id", "content_type", "main_intent", "position_tier", "avg_position",
             "impressions_90d", "clicks_90d", "ctr", "content_age_days", "days_since_last_update"]
print("ONE ROW = ONE PAGE (trailing 90 days)  [ctr is a x100 percentage]")
lane[unit_cols].head(8)

grain check passed: 13,512 rows = 13,512 unique pages

ONE ROW = ONE PAGE (trailing 90 days)  [ctr is a x100 percentage]


,content_id,content_type,main_intent,position_tier,avg_position,impressions_90d,clicks_90d,ctr,content_age_days,days_since_last_update
0,content_304f48230142,keyword article,transactional,striking,10.6,3803,29,0.76,187,20
1,content_a1fb4e703a9e,keyword article,informational,page_3_5,20.3,15320,7,0.05,445,25
2,content_9aa793d4d895,keyword article,informational,page_3_5,36.5,12581,11,0.09,141,20
3,content_331d6c4de07b,keyword article,commercial,page_1,6.2,11751,58,0.49,463,22
4,content_d99b7a2d90ca,keyword article,informational,page_3_5,44.0,19140,24,0.13,263,14
5,content_d4084a4bc775,keyword article,transactional,page_1,8.5,3970,1,0.03,147,20
7,content_a63219c6e95a,keyword article,commercial,page_3_5,21.2,1724,1,0.06,445,22
8,content_5e6c160719bc,keyword article,informational,page_3_5,46.0,32574,29,0.09,90,20


In [5]:
target_sketch = t[["content_id", "position_tier",
                   "impressions_prev_30d", "ctr_prev30", "peer_ctr_prev30", "missed_clicks_prev30",
                   "ctr_next30", "ctr_gap_ratio_next30", "gap_persists_next30"]].round(3)
print("TARGET SKETCH  |  inputs: earlier 30 days  ->  observed target: next 30 days")
target_sketch.sample(8, random_state=7)

TARGET SKETCH  |  inputs: earlier 30 days  ->  observed target: next 30 days


,content_id,position_tier,impressions_prev_30d,ctr_prev30,peer_ctr_prev30,missed_clicks_prev30,ctr_next30,ctr_gap_ratio_next30,gap_persists_next30
9649,content_fb5ea6f6ea7c,page_1,4947,0.000,0.342,16.915,0.054,0.150,1
15191,content_cd6fdaf35403,striking,3748,0.053,0.363,11.623,0.057,0.134,1
2476,content_4c76e9b13aea,page_1,30000,0.097,0.342,73.574,0.057,0.158,1
23292,content_c45e47ff9a30,page_3_5,679,0.000,0.149,1.010,0.000,0.000,1
17045,content_08488289f3c7,striking,1793,0.056,0.363,5.517,0.142,0.335,1
7201,content_bf27c5e56574,page_1,2157,0.278,0.342,1.375,0.198,0.547,0
20594,content_044ba01c048e,striking,956,0.000,0.363,3.475,0.520,1.228,0
24838,content_6dd6d7779405,page_3_5,714,0.000,0.149,1.062,0.193,0.876,0


In [6]:
# What the output looks like for an editor: top of the fixed-rule queue with reason codes
def reasons(r):
    out = []
    if r["position_tier"] in ("top_3", "page_1"):
        out.append("strong position")
    if r["impressions_prev_30d"] >= 3_000:
        out.append("high impressions")
    if r["ctr_prev30"] < 0.5 * r["peer_ctr_prev30"]:
        out.append("CTR < half of peers")
    return ", ".join(out) or "moderate gap"

def action(r):
    if r["position_tier"] in ("top_3", "page_1", "striking"):
        return "rewrite title/meta"
    return "check intent match"

queue = t.sort_values("missed_clicks_prev30", ascending=False).head(10).copy()
queue["rank"] = range(1, len(queue) + 1)
queue["reason_codes"] = queue.apply(reasons, axis=1)
queue["suggested_action"] = queue.apply(action, axis=1)
queue[["rank", "content_id", "position_tier", "main_intent", "impressions_prev_30d",
       "ctr_prev30", "peer_ctr_prev30", "missed_clicks_prev30", "reason_codes",
       "suggested_action"]].round(2)

,rank,content_id,position_tier,main_intent,impressions_prev_30d,ctr_prev30,peer_ctr_prev30,missed_clicks_prev30,reason_codes,suggested_action
26844,1,content_8c19996aa890,top_3,informational,161284,0.24,0.58,544.29,"strong position, high impressions, CTR < half of peers",rewrite title/meta
7678,2,content_8451fc6f034d,top_3,informational,97606,0.03,0.58,534.81,"strong position, high impressions, CTR < half of peers",rewrite title/meta
6653,3,content_5fe46e04994d,page_1,informational,218786,0.11,0.34,498.06,"strong position, high impressions, CTR < half of peers",rewrite title/meta
7445,4,content_c8e9d6ab9013,page_1,informational,111885,0.00,0.34,382.55,"strong position, high impressions, CTR < half of peers",rewrite title/meta
3394,5,content_36ff89c8214e,page_1,informational,106412,0.06,0.34,300.84,"strong position, high impressions, CTR < half of peers",rewrite title/meta
6903,6,content_c84a0ab98e90,page_1,informational,84773,0.02,0.34,270.85,"strong position, high impressions, CTR < half of peers",rewrite title/meta
26531,7,content_cb112fce36be,page_1,transactional,124500,0.14,0.34,254.68,"strong position, high impressions, CTR < half of peers",rewrite title/meta
22028,8,content_73c54f78c06a,page_1,informational,97200,0.09,0.34,248.34,"strong position, high impressions, CTR < half of peers",rewrite title/meta
3070,9,content_91652435f57a,page_1,commercial,75160,0.04,0.34,225.98,"strong position, high impressions, CTR < half of peers",rewrite title/meta
482,10,content_39881853ef0c,page_1,informational,64917,0.01,0.34,215.96,"strong position, high impressions, CTR < half of peers",rewrite title/meta


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The fixed rule is: *"flag a page if its CTR is below half of the typical CTR in its position tier, and rank by missed clicks."* Section 3 showed it is already strong at the very top of the list (the biggest, most stable pages). But the code below shows four reasons it can't be the final answer for the rest of the queue:

1. **Position is continuous, the tier is a bucket.** Inside `page_1` (positions 3–10), median CTR still drops a lot from positions 3–5 to 7–10. The rule treats a page at 3.2 and a page at 9.8 the same, so it over-flags pages at the bottom of each tier and misses ones at the top.
2. **Intent changes what "normal" is.** In the same tier, transactional, commercial and informational pages have different typical CTRs. A single cut-off per tier mixes them.
3. **Clients are very different.** Within the same tier, inside `page_1` pooled CTR per client ranges from about 0.1% to about 0.75%, roughly 8x (brand searches, SERP features like AI overviews or ads). A rule that ignores this flags whole clients instead of specific pages.
4. **The rule is only partly right over time.** Its top 50 hold up well, but across *all* the pages it flags in the earlier window only about 6 in 10 still show the gap 30 days later, and its precision falls as you go further down the list — the rest were one-off dips or noise. The rule has no way to tell those apart; a model trained on the *observed* later-window outcome can learn which signals (volume, stability, age, intent) separate a real gap from noise.

So the expected CTR of a page depends on **several tangled signals at once** (exact position × intent × content type × client × volume). That's too many interacting cut-offs to write by hand and keep honest — which is exactly where ML earns its place. **But** the rule stays as the baseline: a model only replaces it if it beats it on precision@50 on held-out clients (section 3).

In [7]:
print("1) Median CTR inside page_1 by exact position (same tier, different 'normal'):")
p1 = lane[lane["position_tier"] == "page_1"]
print(p1.groupby(pd.cut(p1["avg_position"], [3, 5, 7, 10]), observed=True)["ctr"]
        .agg(["size", "median"]).round(3).to_string())

def pooled(g):
    return 100 * g["clicks_90d"].sum() / g["impressions_90d"].sum()

print("\n2) Pooled CTR % by tier x intent (groups with >= 30 pages):")
ti = (lane.groupby(["position_tier", "main_intent"])
          .apply(lambda g: pd.Series({"pages": len(g), "pooled_ctr": pooled(g)}), include_groups=False))
ti = ti[ti["pages"] >= 30].round(3)
print(ti["pooled_ctr"].unstack().reindex(["top_3", "page_1", "striking", "page_3_5", "deep"]).to_string())

print("\n3) Pooled CTR % per client inside page_1 (clients with >= 20 pages there):")
cc = p1.groupby("client_id").filter(lambda g: len(g) >= 20).groupby("client_id").apply(pooled, include_groups=False)
print(f"   {len(cc)} clients: min {cc.min():.3f}%, median {cc.median():.3f}%, max {cc.max():.3f}%  "
      f"(max/min = {cc.max() / cc.min():.0f}x)")

flag_prev = t["ctr_prev30"] < 0.5 * t["peer_ctr_prev30"]
print(f"\n4) Fixed rule flags {flag_prev.sum():,} pages in the earlier window; "
      f"{t.loc[flag_prev, 'gap_persists_next30'].mean():.1%} still have the gap 30 days later "
      f"(base rate {base_rate:.1%}). The other {1 - t.loc[flag_prev, 'gap_persists_next30'].mean():.0%} "
      "were noise or one-off dips.")

1) Median CTR inside page_1 by exact position (same tier, different 'normal'):
              size  median
avg_position              
(3, 5]        1539    0.33
(5, 7]        2214    0.24
(7, 10]       2381    0.20

2) Pooled CTR % by tier x intent (groups with >= 30 pages):
main_intent    commercial  informational  transactional
position_tier                                          
top_3               0.515          0.396          0.624
page_1              0.328          0.351          0.367
striking            0.405          0.344          0.347
page_3_5            0.196          0.138          0.202
deep                0.030          0.038            NaN

3) Pooled CTR % per client inside page_1 (clients with >= 20 pages there):
   19 clients: min 0.095%, median 0.340%, max 0.750%  (max/min = 8x)

4) Fixed rule flags 4,710 pages in the earlier window; 61.3% still have the gap 30 days later (base rate 37.3%). The other 39% were noise or one-off dips.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous page IDs; client IDs are used for grouping and never printed)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.